# Featuresmith Tutorial: 03 — Understanding the ML Readiness Score

Learn how Featuresmith computes an explainable 0–100 ML Readiness Score across 7 health dimensions (v0.4.0) with transparent mathematical weighting, deduction rules, and actionable fix suggestions.

---


## 1. What is the ML Readiness Score?
The ML Readiness Score answers a fundamental question: *'Is this dataset ready for model training?'*

It translates complex statistical findings into a single, explainable 0–100 score supported by 7 health dimensions:
- **Schema Health**
- **Missing Values**
- **Feature Quality**
- **Distribution Health**
- **Leakage Risk**
- **Data Quality**
- **Consistency**

In v0.4.0 every dimension carries the same default weight of `1.0`, so the overall score is the plain arithmetic mean of the applicable dimension scores. (Per-dimension weight configuration is a documented future capability, not yet configurable.)

### Deduction Rules
Base score per dimension starts at 100. Findings deduct points based on severity:
- **CRITICAL finding**: -30 points
- **WARNING finding**: -15 points
- **INFO finding**: -5 points
Scores are clamped to [0, 100] and rounded to one decimal place.

**Note (v0.4.0):** The Class Balance dimension is omitted pending minority-class detector implementation. Distribution Health reads `BasicStatisticsReviewer` as a stub. Data Quality and Consistency dimensions have been reconciled (cardinality no longer double-counted). See `features/ML-Readiness-Score.md` for details.

### Prerequisite: Prepare the California Housing Dataset
This notebook loads `examples/data/processed/california_housing.csv`, which the example scripts generate and which is **not** bundled in the repository. From the repository root, run the two preparation steps first:

```bash
python examples/download_datasets.py  # network fetch (requires scikit-learn)
python examples/prepare_datasets.py
```


### Step 1: Compute Scorecard on California Housing

In [1]:
import os

import featuresmith as fs

data_path = os.path.join("..", "data", "processed", "california_housing.csv")
dataset = fs.load(data_path)
review_res = fs.review(dataset, target_column="median_house_value")
scorecard = fs.score(review_res)

if scorecard:
    print(f"Overall ML Readiness Score: {scorecard.overall:.1f} / 100\n")
    print(f"{'Dimension':<22} | {'Score':<7} | {'Weight':<6} | Rationale")
    print("-" * 70)
    for dim in scorecard.dimensions:
        print(
            f"{dim.label:<22} | {dim.score:5.1f}   | {dim.weight:4.2f}   | {dim.rationale[:35]}..."
        )

Overall ML Readiness Score: 88.6 / 100

Dimension              | Score   | Weight | Rationale
----------------------------------------------------------------------
Schema Health          | 100.0   | 1.00   | Schema Health scored 100/100 with n...
Missing Values         | 100.0   | 1.00   | Missing Values scored 100/100 with ...
Feature Quality        | 100.0   | 1.00   | Feature Quality scored 100/100 with...
Distribution Health    |  20.0   | 1.00   | Distribution Health scored 20/100; ...
Leakage Risk           | 100.0   | 1.00   | Leakage Risk scored 100/100 with no...
Data Quality           | 100.0   | 1.00   | Data Quality scored 100/100 with no...
Consistency            | 100.0   | 1.00   | Consistency scored 100/100 with no ...


### Step 2: Extract Actionable Fix Suggestions

In [2]:
print("Actionable Fix Suggestions to Improve Score:")
for dim in scorecard.dimensions:
    if dim.suggested_actions:
        print(f"\n[{dim.label}]")
        for action in dim.suggested_actions:
            print(f"  -> {action}")

Actionable Fix Suggestions to Improve Score:

[Distribution Health]
  -> Address the flagged issue: High skewness in column 'average_rooms' (in column 'average_rooms').
  -> Address the flagged issue: High kurtosis in column 'average_rooms' (in column 'average_rooms').
  -> Address the flagged issue: High skewness in column 'average_bedrooms' (in column 'average_bedrooms').
  -> Address the flagged issue: High kurtosis in column 'average_bedrooms' (in column 'average_bedrooms').
  -> Address the flagged issue: High skewness in column 'population' (in column 'population').
  -> Address the flagged issue: High kurtosis in column 'population' (in column 'population').
  -> Address the flagged issue: High skewness in column 'average_occupancy' (in column 'average_occupancy').
  -> Address the flagged issue: High kurtosis in column 'average_occupancy' (in column 'average_occupancy').


### Key Takeaways & Connection to Next Tutorial
- The ML Readiness Score is completely deterministic and reproducible.
- Dimensions currently use uniform `1.0` weights, so the overall score is a simple mean of the applicable dimension scores.
- Fix suggestions provide exact data pipeline remedies.

**Next Tutorial**: In `04_leakage_detection.ipynb`, we dive deep into Intelligent Leakage Detection and the 6 pattern detectors that prevent target leakage bugs.